# Experimento TCC — Correção Automatizada com e sem RAG

Este notebook executa o experimento completo do TCC via API do backend.

**Fluxo:**
1. Setup (registro, login, criação de turma e alunos)
2. Condição A — Com RAG (upload de PDF + publish)
3. Condição B — Sem RAG (publish sem PDF)
4. QA4 — Estabilidade (5 execuções da Condição A)
5. Coleta e exportação dos resultados

## 0. Configuração

In [ ]:
import requests
import json
import time
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

BASE_URL = "http://localhost:8000"
PDF_PATH = Path("../data/test_material.pdf")

assert PDF_PATH.exists(), f"PDF não encontrado em {PDF_PATH.resolve()}"

# Estado global do experimento
state = {}

def api(method, path, **kwargs):
    """Helper para chamadas à API com auth automático."""
    headers = kwargs.pop("headers", {})
    if "token" in state:
        headers["Authorization"] = f"Bearer {state['token']}"
    resp = getattr(requests, method)(f"{BASE_URL}{path}", headers=headers, **kwargs)
    if not resp.ok:
        print(f"❌ {method.upper()} {path} → {resp.status_code}")
        print(resp.text[:500])
    return resp

print("✅ Configuração carregada")
print(f"   Backend: {BASE_URL}")
print(f"   PDF: {PDF_PATH.resolve()}")

## 1. Dados do Experimento

Questões e respostas extraídas de `experiment_A_final_6.json` (dados verificados).

In [ ]:
# ── Questões ──────────────────────────────────────────────────────────────────

QUESTIONS = [
    {
        "id": "Q1",
        "order": 1,
        "points": 10.0,
        "statement": (
            "Diferencie uma `Array` (vetor estático) de uma `Lista Encadeada Simples` "
            "quanto aos seguintes aspectos: alocação de memória, acesso a um elemento "
            "em uma posição arbitrária (índice `k`), e impacto da inserção/remoção de "
            "um elemento no meio da estrutura. Justifique suas respostas com exemplos "
            "de complexidade de tempo O(n)."
        ),
    },
    {
        "id": "Q2",
        "order": 2,
        "points": 10.0,
        "statement": (
            "Compare os algoritmos de ordenação `Merge Sort` e `Quick Sort` considerando "
            "os seguintes critérios: estabilidade, complexidade de tempo no pior caso e "
            "complexidade de espaço. Explique as implicações de cada característica para "
            "a escolha de um algoritmo em cenários específicos."
        ),
    },
    {
        "id": "Q3",
        "order": 3,
        "points": 10.0,
        "statement": (
            "Descreva as propriedades que definem uma `Árvore Binária de Busca (BST)` "
            "balanceada e uma não balanceada. Explique qual a principal vantagem de uma "
            "BST balanceada em relação a uma não balanceada, especificando a complexidade "
            "de tempo de busca, inserção e remoção em ambos os cenários (melhor e pior caso)."
        ),
    },
]

# ── Alunos ────────────────────────────────────────────────────────────────────

STUDENTS = [
    {"name": "Aluno 1 (Excelente)",  "email": "aluno1.excelente@tcc.test",  "profile": "excellent"},
    {"name": "Aluno 2 (Adequado)",   "email": "aluno2.adequado@tcc.test",   "profile": "average"},
    {"name": "Aluno 3 (Fraco)",      "email": "aluno3.fraco@tcc.test",      "profile": "poor"},
    {"name": "Aluno 4 (Fora do Tema)", "email": "aluno4.foratema@tcc.test", "profile": "off_topic"},
]

# ── Respostas (12 = 3 questões × 4 alunos) ──────────────────────────────────
# Chave: (question_order, student_index)

ANSWERS = {
    # ── Q1: Array vs Lista Encadeada ──
    (1, 0): (
        "Arrays alocam memória contígua; Listas Encadeadas alocam nós não contiguamente. "
        "O acesso a um elemento `k` em Array é O(1) (direto por índice); em Lista é O(n), "
        "pois exige travessia sequencial. A inserção/remoção no meio em Array é O(n), devido "
        "ao deslocamento de elementos. Em Lista Encadeada, também é O(n) para localizar a "
        "posição e ajustar ponteiros."
    ),
    (1, 1): (
        "Uma Array aloca memória contígua, permitindo acesso direto a um elemento `k`. Já a "
        "Lista Encadeada Simples aloca nós dispersos, sendo o acesso a `k` feito por travessia, "
        "resultando em complexidade O(n). Inserir ou remover no meio de uma Array exige deslocar "
        "elementos, o que também custa O(n). Na Lista Encadeada, após achar o ponto, a "
        "inserção/remoção exige ajustar os ponteiros, podendo também levar tempo O(n) para localizar."
    ),
    (1, 2): (
        "`Arrays` alocam memória de forma estática em segmentos contíguos, facilitando o acesso "
        "O(1) a qualquer elemento via índice `k` por meio de um cálculo direto. Já `Listas "
        "Encadeadas` utilizam alocação dinâmica, onde cada nó armazena dados e ponteiros, o que "
        "exige um acesso O(n) para encontrar um elemento em `k`. A inserção/remoção no meio da "
        "`Array` causa um grande impacto de O(n) devido à necessidade de reorganização de todos "
        "os elementos, enquanto na `Lista` é mais eficiente, pois apenas ponteiros são alterados, "
        "mas a busca por esse ponto ainda é O(n)."
    ),
    (1, 3): (
        "A diferenciação entre planetas rochosos e gigantes gasosos reside na sua composição "
        "interna e formação. Enquanto rochosos possuem núcleo sólido e superfícies definidas, "
        "como a Terra, gigantes gasosos, como Júpiter, são vastas esferas de hidrogênio e hélio, "
        "sem superfície sólida clara. A observação de exoplanetas pode variar, revelando atmosferas "
        "ou a ausência delas, impactando a busca por vida."
    ),

    # ── Q2: Merge Sort vs Quick Sort ──
    (2, 0): (
        "Merge Sort (MS) é estável, O(n log n) tempo no pior caso e O(n) espaço auxiliar, ideal "
        "para estabilidade e desempenho garantido.\n"
        "Quick Sort (QS) é instável, O(n²) tempo no pior caso e O(log n) espaço médio (pilha "
        "recursiva), favorecendo eficiência de espaço.\n"
        "A estabilidade do MS é crucial para manter a ordem relativa de elementos iguais, e sua "
        "complexidade de tempo garantida é vital em sistemas críticos.\n"
        "O QS, embora mais rápido na prática para muitos cenários, exige boa escolha de pivô "
        "para evitar seu pior caso de tempo."
    ),
    (2, 1): (
        "Merge Sort é tipicamente estável e tem um desempenho de tempo mais previsível, sendo "
        "O(n log n) no pior caso, mas utiliza mais espaço auxiliar (O(n)). Quick Sort geralmente "
        "não é estável e pode ter um pior caso de tempo de O(n^2).\n\n"
        "No entanto, Quick Sort costuma ser mais eficiente em termos de espaço (O(log n) na "
        "média). Assim, escolhe-se Merge Sort quando a ordem relativa de elementos iguais é "
        "crucial ou a previsibilidade do tempo é prioritária.\n\n"
        "Quick Sort é preferível para menor uso de memória, mas é preciso considerar a variação "
        "do seu desempenho dependendo dos dados de entrada."
    ),
    (2, 2): (
        "Merge Sort é instável e tem complexidade de tempo de O(N^2) no pior caso, gastando "
        "pouco espaço. O Quick Sort é estável e mais eficiente, com O(N log N) no pior caso, "
        "mas exige mais memória. A escolha depende se a performance ou a estabilidade é a "
        "prioridade, sendo o Quick Sort melhor para dados já ordenados."
    ),
    (2, 3): (
        "Grelhar e assar diferem muito. Grelhar oferece sabor defumado intenso e cozimento "
        "rápido para carnes finas, mas requer atenção constante. Assar distribui calor "
        "uniformemente, ideal para bolos e pães, garantindo maciez interna. A complexidade "
        "do preparo e os equipamentos variam. A escolha impacta diretamente a textura final "
        "e a suculência do prato."
    ),

    # ── Q3: Árvore Binária de Busca ──
    (3, 0): (
        "Uma BST balanceada garante altura O(log n) (diferença de altura entre subárvores "
        "limitada); a não balanceada pode degenerar para O(n). A principal vantagem da "
        "balanceada é manter busca, inserção e remoção em O(log n) no melhor e pior caso. "
        "Já na não balanceada, estas operações são O(log n) no melhor caso, mas O(n) no "
        "pior caso (árvore degenerada), desempenho que a balanceada evita."
    ),
    (3, 1): (
        "Árvores balanceadas mantêm uma altura proporcional ao logaritmo do número de "
        "elementos, assegurando desempenho consistente O(log n) para busca, inserção e "
        "remoção no melhor e pior caso. Já as não balanceadas podem ter altura linear, "
        "degenerando para um pior caso O(n) nessas operações. A principal vantagem das "
        "balanceadas é a garantia de eficiência, evitando o desempenho imprevisível do "
        "pior caso linear das não balanceadas."
    ),
    (3, 2): (
        "Uma BST balanceada tem seus nós organizados de forma que todos os caminhos sejam "
        "de igual comprimento e cada nó filho é sempre maior que o pai, o que a torna "
        "otimizada. Uma não balanceada não segue essa regra, podendo virar uma lista. A "
        "vantagem principal da balanceada é sua agilidade. A complexidade de busca, inserção "
        "e remoção é O(log n) para a balanceada (todos os casos), e para a não balanceada, "
        "é O(1) no melhor e O(n!) no pior."
    ),
    (3, 3): (
        "Um pão balanceado requer levedura, farinha e água em proporções exatas para uma "
        "massa leve e aerada. Um pão não balanceado resulta em um miolo denso ou seco. A "
        "principal vantagem do balanceado é a melhor fermentação e sabor; o tempo de sova "
        "é O(1) no melhor caso, ou O(N) para ajustes no pior."
    ),
}

print(f"✅ Dados carregados: {len(QUESTIONS)} questões, {len(STUDENTS)} alunos, {len(ANSWERS)} respostas")

## 2. Setup — Registro, Login, Turma e Alunos

In [ ]:
# ── 2.1 Registro do professor ─────────────────────────────────────────────────

TEACHER = {
    "first_name": "Professor",
    "last_name": "TCC",
    "email": f"professor.tcc.{int(time.time())}@test.com",
    "password": "SenhaSegura@123",
    "user_type": "teacher",
}

r = api("post", "/users/register", json=TEACHER)
r.raise_for_status()
teacher_data = r.json()
state["teacher_uuid"] = teacher_data["uuid"]
print(f"✅ Professor registrado: {state['teacher_uuid']}")

In [ ]:
# ── 2.2 Login ─────────────────────────────────────────────────────────────────

r = api("post", "/auth/login", json={
    "email": TEACHER["email"],
    "password": TEACHER["password"],
})
r.raise_for_status()
state["token"] = r.json()["access_token"]
print(f"✅ Login OK — token obtido")

In [ ]:
# ── 2.3 Criar turma ──────────────────────────────────��───────────────────────

r = api("post", "/classes", json={
    "name": "Algoritmos e Estrutura de Dados — TCC Experimento",
    "description": "Turma experimental para validação do sistema de correção automática",
    "year": 2026,
    "semester": 1,
})
r.raise_for_status()
state["class_uuid"] = r.json()["uuid"]
print(f"✅ Turma criada: {state['class_uuid']}")

In [ ]:
# ── 2.4 Adicionar alunos ─────────────────────────────────────────────────────

r = api("post", f"/classes/{state['class_uuid']}/students", json={
    "students": [
        {"full_name": s["name"], "email": s["email"]}
        for s in STUDENTS
    ]
})
r.raise_for_status()
students_resp = r.json()

# Mapear índice do aluno ao UUID retornado
# A API retorna os alunos na ordem enviada
student_list = students_resp.get("students", students_resp.get("data", []))
state["student_uuids"] = [s["uuid"] for s in student_list]

for i, s in enumerate(student_list):
    print(f"  {STUDENTS[i]['profile']:>10} → {s['uuid']}")
print(f"\n✅ {len(student_list)} alunos adicionados")

## 3. Helper — Criar prova, questões, respostas e publicar

Função reutilizável para as condições A, B e QA4.

In [ ]:
def run_experiment(label: str, upload_pdf: bool = True, poll_interval: int = 10, max_wait: int = 600):
    """
    Executa um ciclo completo do experimento:
    1. Cria prova
    2. Cria questões
    3. Submete respostas
    4. (Opcional) Upload de PDF para RAG
    5. Publica prova (dispara correção)
    6. Aguarda conclusão (polling)
    7. Coleta resultados (incluindo C1/C2/árbitro)
    
    Returns:
        dict com exam_uuid, resultados por aluno/questão e metadados
    """
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"\n{'='*70}")
    print(f"  {label} — início {ts}")
    print(f"  RAG: {'SIM' if upload_pdf else 'NÃO'}")
    print(f"{'='*70}\n")
    
    # ── Criar prova ───────────────────────────────────────────────────────
    now = datetime.now()
    r = api("post", "/exams", json={
        "title": f"{label} — {now.strftime('%Y%m%d_%H%M%S')}",
        "description": f"Experimento TCC: {label}",
        "class_uuid": state["class_uuid"],
        "starts_at": now.isoformat(),
        "ends_at": (now + timedelta(hours=2)).isoformat(),
    })
    r.raise_for_status()
    exam_uuid = r.json()["uuid"]
    print(f"  📝 Prova criada: {exam_uuid}")

    # ── Criar questões ────────────────────────────────────────────────────
    question_uuids = []
    for q in QUESTIONS:
        r = api("post", "/exam-questions", json={
            "exam_uuid": exam_uuid,
            "statement": q["statement"],
            "question_order": q["order"],
            "points": q["points"],
        })
        r.raise_for_status()
        question_uuids.append(r.json()["uuid"])
    print(f"  📋 {len(question_uuids)} questões criadas")

    # ── Submeter respostas ────────────────────────────────────────────────
    answer_count = 0
    for (q_order, s_idx), answer_text in ANSWERS.items():
        r = api("post", "/student-answers", json={
            "exam_uuid": exam_uuid,
            "question_uuid": question_uuids[q_order - 1],
            "student_uuid": state["student_uuids"][s_idx],
            "answer_text": answer_text,
        })
        r.raise_for_status()
        answer_count += 1
    print(f"  ✏️  {answer_count} respostas submetidas")

    # ── Upload PDF (RAG) ──────────────────────────────────────────────────
    if upload_pdf:
        with open(PDF_PATH, "rb") as f:
            r = api("post", f"/attachments/upload",
                    params={"exam_uuid": exam_uuid},
                    files={"file": (PDF_PATH.name, f, "application/pdf")})
        r.raise_for_status()
        print(f"  📄 PDF enviado para RAG")
    else:
        print(f"  📄 Sem PDF — condição sem RAG")

    # ── Publicar (dispara correção em background) ─────────────────────────
    r = api("post", f"/exams/{exam_uuid}/publish")
    r.raise_for_status()
    print(f"  🚀 Prova publicada — correção iniciada")

    # ── Polling até GRADED ────────────────────────────────────────────────
    start = time.time()
    while time.time() - start < max_wait:
        time.sleep(poll_interval)
        r = api("get", f"/exams/{exam_uuid}")
        r.raise_for_status()
        status = r.json().get("status", "")
        elapsed = int(time.time() - start)
        print(f"  ⏳ {elapsed}s — status: {status}", end="\r")
        if status in ("GRADED", "WARNING"):
            print(f"  ✅ Correção finalizada em {elapsed}s — status: {status}")
            break
    else:
        print(f"  ⚠️ Timeout ({max_wait}s) — verificar manualmente")

    # ── Coletar resultados via review ─────────────────────────────────────
    r = api("get", f"/reviews/exams/{exam_uuid}")
    r.raise_for_status()
    review = r.json()

    # Estruturar resultados (agora com C1/C2/árbitro)
    results = []
    for q_data in review.get("questions", []):
        for ans in q_data.get("student_answers", q_data.get("answers", [])):
            results.append({
                "exam_uuid": exam_uuid,
                "label": label,
                "rag": upload_pdf,
                "question": q_data.get("statement", "")[:60],
                "question_uuid": q_data.get("question_uuid", q_data.get("uuid")),
                "student_name": ans.get("student_name", ""),
                "student_uuid": ans.get("student_uuid"),
                "score": ans.get("score"),
                "c1_score": ans.get("c1_score"),
                "c2_score": ans.get("c2_score"),
                "arbiter_score": ans.get("arbiter_score"),
                "divergence_detected": ans.get("divergence_detected", False),
                "divergence_value": ans.get("divergence_value"),
                "consensus_method": ans.get("consensus_method"),
                "criteria_scores": ans.get("criteria_scores", []),
                "agent_criteria_scores": ans.get("agent_criteria_scores", []),
                "feedback": ans.get("feedback", ""),
            })
    
    df = pd.DataFrame(results)
    print(f"\n  📊 {len(results)} resultados coletados")
    if len(df) > 0:
        print(f"  📈 Média geral: {df['score'].mean():.2f}")
        n_div = df["divergence_detected"].sum()
        if n_div > 0:
            print(f"  ⚠️  {n_div} divergências detectadas (árbitro acionado)")
    
    return {
        "label": label,
        "exam_uuid": exam_uuid,
        "rag": upload_pdf,
        "results": results,
        "df": df,
        "raw_review": review,
    }

## 4. Condição A — Com RAG

In [ ]:
cond_a = run_experiment("Condição A (com RAG)", upload_pdf=True)

In [ ]:
cond_a["df"][["student_name", "question", "c1_score", "c2_score", "arbiter_score", "score", "consensus_method", "divergence_detected"]]

## 5. Condição B — Sem RAG

In [ ]:
cond_b = run_experiment("Condição B (sem RAG)", upload_pdf=False)

In [ ]:
cond_b["df"][["student_name", "question", "c1_score", "c2_score", "arbiter_score", "score", "consensus_method", "divergence_detected"]]

## 6. Comparação A vs B

In [ ]:
# Pivot para comparação
def build_comparison(cond_a_df, cond_b_df):
    a = cond_a_df[["student_name", "question", "score"]].rename(columns={"score": "Com RAG"})
    b = cond_b_df[["student_name", "question", "score"]].rename(columns={"score": "Sem RAG"})
    merged = a.merge(b, on=["student_name", "question"])
    merged["Delta"] = merged["Com RAG"] - merged["Sem RAG"]
    return merged

comparison = build_comparison(cond_a["df"], cond_b["df"])
comparison

In [ ]:
# Resumo por nível de qualidade
profile_map = {s["name"]: s["profile"] for s in STUDENTS}
comparison["profile"] = comparison["student_name"].map(profile_map)

summary = comparison.groupby("profile").agg({
    "Com RAG": "mean",
    "Sem RAG": "mean",
    "Delta": "mean",
}).round(2)

print("Média por nível de qualidade:")
summary

## 7. QA4 — Estabilidade (5 execuções com RAG)

In [ ]:
QA4_RUNS = 5
qa4_results = []

for i in range(QA4_RUNS):
    result = run_experiment(f"QA4 Run {i+1}/{QA4_RUNS}", upload_pdf=True)
    qa4_results.append(result)
    print(f"\n{'─'*70}\n")

print(f"\n✅ QA4 completo — {QA4_RUNS} execuções realizadas")

In [ ]:
# Consolidar QA4 — variância por aluno/questão
import numpy as np

all_qa4 = pd.concat([r["df"] for r in qa4_results], ignore_index=True)

qa4_stats = all_qa4.groupby(["student_name", "question"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std"),
    min_score=("score", "min"),
    max_score=("score", "max"),
    var_score=("score", "var"),
).round(2)

print(f"Variância média: {qa4_stats['var_score'].mean():.2f}")
print(f"Desvio padrão médio: {qa4_stats['std_score'].mean():.2f}")
print()
qa4_stats

In [ ]:
# Resumo QA4 por nível
all_qa4["profile"] = all_qa4["student_name"].map(profile_map)

qa4_by_profile = all_qa4.groupby("profile").agg(
    mean_score=("score", "mean"),
    std_score=("score", "std"),
    var_score=("score", "var"),
).round(2)

print("QA4 — Estabilidade por nível:")
qa4_by_profile

## 8. Exportar Resultados

In [ ]:
# Salvar tudo em JSON rastreável
output_dir = Path("../tcc/results")
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def export_run(run_data, filename):
    export = {
        "label": run_data["label"],
        "exam_uuid": run_data["exam_uuid"],
        "rag": run_data["rag"],
        "timestamp": timestamp,
        "results": run_data["results"],
    }
    path = output_dir / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(export, f, ensure_ascii=False, indent=2)
    print(f"  💾 {path}")

# Condição A e B
export_run(cond_a, f"notebook_cond_A_{timestamp}.json")
export_run(cond_b, f"notebook_cond_B_{timestamp}.json")

# QA4 — todas as runs em um arquivo
qa4_export = {
    "label": "QA4 Estabilidade",
    "num_runs": QA4_RUNS,
    "timestamp": timestamp,
    "runs": [
        {
            "run": i + 1,
            "exam_uuid": r["exam_uuid"],
            "results": r["results"],
        }
        for i, r in enumerate(qa4_results)
    ],
}
qa4_path = output_dir / f"notebook_QA4_{timestamp}.json"
with open(qa4_path, "w", encoding="utf-8") as f:
    json.dump(qa4_export, f, ensure_ascii=False, indent=2)
print(f"  💾 {qa4_path}")

print(f"\n✅ Todos os resultados exportados para {output_dir}/")

In [ ]:
# Resumo final
print("="*70)
print("  RESUMO DO EXPERIMENTO")
print("="*70)
print(f"\n  Condição A (com RAG):  média = {cond_a['df']['score'].mean():.2f}")
print(f"  Condição B (sem RAG):  média = {cond_b['df']['score'].mean():.2f}")
print(f"  Delta médio (A - B):   {comparison['Delta'].mean():.2f}")
print(f"\n  QA4 — {QA4_RUNS} execuções:")
print(f"    Variância média:     {qa4_stats['var_score'].mean():.2f}")
print(f"    Desvio padrão médio: {qa4_stats['std_score'].mean():.2f}")
print(f"\n  Arquivos salvos em: {output_dir.resolve()}/")
print("="*70)